# UZH-FPV + Spyx New Architectures

This notebook targets the UZH-FPV Drone Racing dataset at https://fpv.ifi.uzh.ch/.

Status in Tonic:
- No direct `tonic.datasets` class for UZH-FPV in this environment.

Research focus:
- event-based racing-state estimation
- architecture ablations with newer Spyx models
- small Optuna search over event encoder families


In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import optuna

import spyx.models as fm

DATA_ROOT = Path("../data/UZH_FPV")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_uzh_fpv():
    import tonic.datasets as d
    names = {n.lower() for n in dir(d)}
    return any(k in names for k in ["uzhfpv", "fpv", "uzh_fpv"])

print("tonic_has_uzh_fpv =", tonic_has_uzh_fpv())
print("expected dirs from uzh_fpv_open:", ["calib", "raw", "output"])


In [ ]:
def synthetic_event_batch(batch=4, T=16, H=64, W=64, C=2, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.poisson(0.02, size=(T, batch, H, W, C)).astype(np.float32)
    return jnp.asarray(x)


def make_architectures(input_hw=(64, 64), input_channels=2, output_dim=6):
    conv_cfg = fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=output_dim)
    sparse_cfg = fm.SparseConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=output_dim, event_threshold=0.0)
    dw_cfg = fm.DepthwiseSepConvConfig(input_hw=input_hw, input_channels=input_channels, depth_multiplier1=1, pointwise1=24, depth_multiplier2=1, pointwise2=48, output_dim=output_dim)
    return {
        "conv_lif": lambda x: fm.ConvLIFSNN(conv_cfg)(x),
        "ternary_conv_lif": lambda x: fm.TernaryConvLIFSNN(conv_cfg)(x),
        "sparse_event_conv_lif": lambda x: fm.SparseEventConvLIFSNN(sparse_cfg)(x),
        "depthwise_sep_conv_lif": lambda x: fm.DepthwiseSeparableConvLIFSNN(dw_cfg)(x),
    }


x = synthetic_event_batch()
for name, fn in make_architectures().items():
    net = hk.without_apply_rng(hk.transform(fn))
    params = net.init(jax.random.PRNGKey(0), x)
    pred, aux = net.apply(params, x)
    print(name, pred.shape, np.asarray(aux["spike_rate"]))


## Optuna Sweep

This study treats UZH-FPV as an event-regression benchmark and explores compact Spyx encoders with a spike-rate penalty. Swap the synthetic batch with cached mDAVIS event tensors once local data is available.


In [ ]:
def uzh_fpv_objective(trial):
    arch = trial.suggest_categorical("arch", ["conv", "ternary", "sparse", "depthwise"])
    channels1 = trial.suggest_categorical("channels1", [16, 24, 32])
    channels2 = trial.suggest_categorical("channels2", [32, 48, 64])
    beta = trial.suggest_float("beta", 0.75, 0.98)
    threshold = trial.suggest_float("threshold", 0.6, 1.4)

    x_local = synthetic_event_batch(seed=trial.number)
    target = jnp.zeros((x_local.shape[1], 6), dtype=jnp.float32)

    if arch == "depthwise":
        cfg = fm.DepthwiseSepConvConfig(input_hw=(64, 64), input_channels=2, depth_multiplier1=1, pointwise1=channels1, depth_multiplier2=1, pointwise2=channels2, output_dim=6, beta=beta, threshold=threshold)
        forward = lambda xb: fm.DepthwiseSeparableConvLIFSNN(cfg)(xb)
    elif arch == "sparse":
        cfg = fm.SparseConvConfig(input_hw=(64, 64), input_channels=2, channels1=channels1, channels2=channels2, output_dim=6, beta=beta, threshold=threshold, event_threshold=0.0)
        forward = lambda xb: fm.SparseEventConvLIFSNN(cfg)(xb)
    else:
        cfg = fm.ConvConfig(input_hw=(64, 64), input_channels=2, channels1=channels1, channels2=channels2, output_dim=6, beta=beta, threshold=threshold)
        forward = (lambda xb: fm.TernaryConvLIFSNN(cfg)(xb)) if arch == "ternary" else (lambda xb: fm.ConvLIFSNN(cfg)(xb))

    net = hk.without_apply_rng(hk.transform(forward))
    params = net.init(jax.random.PRNGKey(0), x_local)
    pred, aux = net.apply(params, x_local)
    mse = jnp.mean((pred - target) ** 2)
    spike_penalty = 0.01 * jnp.mean(jnp.asarray(aux["spike_rate"]))
    return float(mse + spike_penalty)


study = optuna.create_study(direction="minimize", study_name="uzh_fpv_spyx_arch_search")
study.optimize(uzh_fpv_objective, n_trials=8)
study.best_trial.params, study.best_value


## Next Steps for Real UZH-FPV Runs

1. Place real dataset content in `../data/UZH_FPV` with `calib`, `raw`, and `output` folders.
2. Convert mDAVIS events into cached frame tensors or a direct event iterator.
3. Replace synthetic targets with pose or gate-progression targets and rerun the Optuna study.
